# Part 2: The Transformer

This is the core of the workshop. You'll write the full GPT model architecture from scratch in PyTorch.

A GPT is an **autoregressive language model**: given a sequence of tokens, it predicts the next one. Stack this prediction in a loop and you get text generation.

The architecture is a stack of identical **transformer blocks**, each containing:
1. **Multi-head self-attention** — lets each token look at all previous tokens
2. **Feed-forward network (MLP)** — processes each position independently
3. **Residual connections** — add the input back to the output of each sub-layer
4. **Layer normalization** — stabilizes training

By the end of this notebook you'll have a working `GPT` class ready to train.

## Setup

In [ ]:
!pip install -q torch

import torch
import torch.nn as nn
from dataclasses import dataclass

print(f"PyTorch version: {torch.__version__}")

## Step 1: Configuration

`GPTConfig` holds all the hyperparameters that define the model architecture.

- `vocab_size`: comes from the tokenizer — 65 unique characters in Shakespeare
- `block_size`: the context window — how many tokens the model can see at once
- `n_embd`: the width of the model — every hidden state is a vector of this size

In [ ]:
@dataclass
class GPTConfig:
    vocab_size: int = 65       # character-level: 65 unique chars in Shakespeare
    block_size: int = 256      # max sequence length (context window)
    n_layer: int = 6           # number of transformer blocks
    n_head: int = 6            # number of attention heads
    n_embd: int = 384          # embedding dimension

config = GPTConfig()
print(config)

## Step 2: Causal Self-Attention

Self-attention is the mechanism that lets each token attend to (look at) every previous token in the sequence.

The key steps:
1. **Q, K, V projections**: project the input into Query, Key, and Value matrices
2. **Multi-head reshape**: split into `n_head` parallel attention heads
3. **Scaled dot-product attention**: `softmax(QKᵀ / √head_dim) @ V`
4. **Causal masking** (`is_causal=True`): position `i` can only attend to positions `0..i`
5. **Output projection**: combine heads and project back to `n_embd` dimensions

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd)  # Q, K, V projections
        self.c_proj = nn.Linear(config.n_embd, config.n_embd)       # output projection
        self.n_head = config.n_head
        self.n_embd = config.n_embd

    def forward(self, x):
        B, T, C = x.shape
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.n_embd, dim=2)

        # reshape for multi-head: (B, T, C) → (B, n_head, T, head_dim)
        head_dim = C // self.n_head
        q = q.view(B, T, self.n_head, head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_head, head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_head, head_dim).transpose(1, 2)

        # attention with causal mask (each token can only attend to previous tokens)
        y = torch.nn.functional.scaled_dot_product_attention(
            q, k, v, is_causal=True
        )

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)

### Why Multi-Head?

With 6 heads of 64 dimensions each (instead of one head of 384 dimensions), the model can simultaneously track different relationships: one head might track which vowels follow consonants, another might track line-break patterns, another might focus on recent context.

In [ ]:
# Quick sanity check: attention module shapes
attn = CausalSelfAttention(config)
x = torch.randn(2, 16, config.n_embd)  # batch=2, seq_len=16
out = attn(x)
print(f"Input shape:  {x.shape}")
print(f"Output shape: {out.shape}")   # should match input shape

## Step 3: MLP Block

The MLP is applied independently to each position. It expands the representation to 4× the embedding dimension, applies a non-linearity (GELU), and projects back down.

This is where the model does most of its "thinking" — the attention gathers information, the MLP processes it.

**Why GELU instead of ReLU?** GELU (Gaussian Error Linear Unit) is smoother than ReLU. It doesn't have a hard cutoff at zero, which helps gradient flow. GPT-2 uses the `tanh` approximation for speed.

In [ ]:
class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd)
        self.gelu = nn.GELU(approximate='tanh')
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd)

    def forward(self, x):
        x = self.c_fc(x)       # project up: 384 → 1536
        x = self.gelu(x)       # non-linearity
        return self.c_proj(x)  # project back down: 1536 → 384

## Step 4: Transformer Block

Each block wraps attention and MLP with two key design choices:

1. **Pre-norm** (LayerNorm before attention/MLP, not after): stabilizes training by normalizing inputs to each sub-layer
2. **Residual connections** (`x = x + sublayer(x)`): lets gradients flow directly through the network, making deep networks trainable

In [ ]:
class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.n_embd)
        self.attn = CausalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.n_embd)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))   # attention with residual connection
        x = x + self.mlp(self.ln_2(x))    # MLP with residual connection
        return x

## Step 5: The Full GPT Model

Now assemble all the pieces:

- **`wte`** (word token embedding): maps each token ID to a learned vector. Size: `[65, 384]`
- **`wpe`** (word position embedding): maps each position to a learned vector. Size: `[256, 384]`
- **`h`**: a stack of 6 transformer blocks
- **`ln_f`**: final layer norm before the output projection
- **`lm_head`**: linear layer projecting from `n_embd` → `vocab_size` (the next-token logits)

**Weight tying**: the same matrix that maps tokens → embeddings is reused (transposed) to map embeddings → logits at the output. This reduces parameters and improves training — the model's input and output representations of tokens are forced to be consistent.

In [ ]:
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),   # token embeddings
            wpe = nn.Embedding(config.block_size, config.n_embd),   # position embeddings
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = nn.LayerNorm(config.n_embd),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        # weight tying: the output projection shares weights with the token embeddings
        self.transformer.wte.weight = self.lm_head.weight

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(0, T, device=idx.device)

        tok_emb = self.transformer.wte(idx)    # (B, T, n_embd)
        pos_emb = self.transformer.wpe(pos)    # (T, n_embd)
        x = tok_emb + pos_emb                  # (B, T, n_embd) — broadcasting adds position info

        for block in self.transformer.h:
            x = block(x)

        x = self.transformer.ln_f(x)
        logits = self.lm_head(x)               # (B, T, vocab_size)

        loss = None
        if targets is not None:
            loss = nn.functional.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1)
            )
        return logits, loss

## Parameter Count

Where do the parameters live?
- Token embeddings: `65 × 384 = 25K` (tiny with char-level vocab — shared with lm_head)
- Position embeddings: `256 × 384 = 98K`
- Per transformer block: ~1.8M (attention: 4 × 384² = 590K, MLP: 2 × 384 × 1536 = 1.2M)
- 6 blocks: ~10.6M
- Total: ~10.8M

In [ ]:
# Instantiate and count parameters
model = GPT(config)
n_params = sum(p.numel() for p in model.parameters())
print(f"Total parameters: {n_params / 1e6:.1f}M")

# Break down by component
print("\nParameter breakdown:")
for name, module in model.named_modules():
    params = sum(p.numel() for p in module.parameters(recurse=False))
    if params > 0:
        print(f"  {name:<40} {params:>10,}")

In [ ]:
# Test the forward pass
batch_size = 4
seq_len = 32

idx = torch.randint(0, config.vocab_size, (batch_size, seq_len))
targets = torch.randint(0, config.vocab_size, (batch_size, seq_len))

logits, loss = model(idx, targets)
print(f"Input shape:   {idx.shape}")
print(f"Logits shape:  {logits.shape}")
print(f"Loss (random): {loss.item():.4f}  (expected ~{torch.log(torch.tensor(config.vocab_size)):.2f} = ln({config.vocab_size}))")

## Model Configurations

Compare different model sizes from the workshop:

In [ ]:
configs = [
    ("Tiny",   dict(n_layer=2, n_head=2, n_embd=128)),
    ("Small",  dict(n_layer=4, n_head=4, n_embd=256)),
    ("Medium", dict(n_layer=6, n_head=6, n_embd=384)),
]

print(f"{'Name':<10} {'n_layer':>8} {'n_head':>8} {'n_embd':>8} {'Params':>12}")
print("-" * 50)
for name, kwargs in configs:
    cfg = GPTConfig(**kwargs)
    m = GPT(cfg)
    p = sum(p.numel() for p in m.parameters())
    marker = " ← default" if name == "Medium" else ""
    print(f"{name:<10} {cfg.n_layer:>8} {cfg.n_head:>8} {cfg.n_embd:>8} {p/1e6:>11.1f}M{marker}")

## Key Takeaways

- A GPT is a stack of identical transformer blocks
- Each block: LayerNorm → Self-Attention → Residual → LayerNorm → MLP → Residual
- Self-attention lets tokens look at all previous tokens (causal masking prevents looking ahead)
- Multi-head attention runs multiple attention patterns in parallel
- Residual connections and layer norm make deep networks trainable
- Weight tying between input embeddings and output projection reduces parameters

**Next: [Part 3 — The Training Loop](03-training-loop.ipynb)**